In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU 개수:", torch.cuda.device_count())
else:
    print("현재 PyTorch는 GPU를 사용할 수 없습니다.")

PyTorch: 2.14.1+cu126
CUDA 사용 가능: True
GPU: NVIDIA GeForce GTX 1060 6GB
GPU 개수: 1


In [2]:
import warnings
import os
from huggingface_hub import login

from dotenv import load_dotenv

warnings.simplefilter(action='ignore')
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### 모델 정보 보기

| 모델 | 구조 | 대표 모델 | 사용 |
|--|--|--|--|
|Encoder-only | 인코더만 사용 | BERT | 입력 문장을 이해하고 분석하는 데 강함 |
|Decoder-only | 디코더만 사용 | GPT | 새로운 문장을 생성하는 데 강함 |
|Encoder + Decoder | 둘다 사용 | T5 | 둘 다 |

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# bert : 인코더만 사용
info = api.model_info("klue/bert-base")

print("task : ", info.pipeline_tag)
print("다운로드 : ", info.downloads)
print("라이선스 : ", info.card_data.get('license'))

if info.safetensors:
    print("파라미터 수 : ", info.safetensors.total)


task :  fill-mask
다운로드 :  75750
라이선스 :  cc-by-sa-4.0
파라미터 수 :  111243522


In [2]:
for m in api.list_models(filter='text-classification',search='korean',sort='downloads', limit=10):
    print(f"{(m.downloads or 0):>10,} {m.id}")

     2,879 nlp04/korean_sentiment_analysis_kcelectra
     2,202 cringepnh/koelectra-korean-sentiment
       801 matthewburke/korean_sentiment
       536 WhitePeak/bert-base-cased-Korean-sentiment
        92 Seonghaa/korean-emotion-classifier-roberta
        78 hyun5ooo/deberta-v3-base-korean-ai-detect
        77 beomi/korean-hatespeech-multilabel
        48 GGARA02/kcelectra-korean-emotion
        47 beomi/korean-hatespeech-classifier
        44 2tle/korean-curse-detection


### 파이프라인
`pipeline('태스크명',model='모델id')`

#### 1) 감정분석

In [2]:
from transformers import pipeline

clf = pipeline('sentiment-analysis', model='distilbert/distilbert-base-uncased-finetuned-sst-2-english',device_map="auto",dtype=torch.float16)

print(clf(['I love this phone.','The battery dies too fast.']))
print(clf(['배송이 너무 늦어요.']))

c:\source\llm3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 367.14it/s]


[{'label': 'POSITIVE', 'score': 0.9998155236244202}, {'label': 'NEGATIVE', 'score': 0.9996964931488037}]
[{'label': 'POSITIVE', 'score': 0.5610645413398743}]


2) 빈칸 채우기 - 한국어 BERT

In [4]:
clf = pipeline('fill-mask', model='klue/bert-base',device_map="auto",dtype=torch.float16)

#for r in clf("대한민국의 수도는 [MASK]이다."):
for r in clf("오늘의 점심은 [MASK]이다."):
    print(r['token_str'], round(r['score'], 3))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 492.17it/s]
[transformers] BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


치킨 0.061
삼겹살 0.042
비빔밥 0.04
짜장면 0.024
도시락 0.023


#### 3) 제로샷
- 학습할 때 없었던 label 설정 가능

In [8]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",device_map="auto",dtype=torch.float16)

sequence_to_classify = "Angela Merkel ist eine Politikerin in Deutschland und Vorsitzende der CDU"
candidate_labels = ["politics", "economy", "entertainment", "environment"]
output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
# print(output)
print(list(zip(output['labels'], [round(s, 3) for s in output['scores']])))


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 686.88it/s]


[('politics', 0.966), ('economy', 0.023), ('environment', 0.007), ('entertainment', 0.004)]


In [9]:
sequence_to_classify = "손흥민이 시즌 10호 골을 터뜨렸다."
candidate_labels = ["경제", "스포츠", "연예", "정치"]
output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
# print(output)
print(list(zip(output['labels'], [round(s, 3) for s in output['scores']])))

[('스포츠', 0.501), ('연예', 0.288), ('경제', 0.108), ('정치', 0.103)]


#### ner(named entity recognition)
- 개체명 인식

In [14]:
ner = pipeline("ner", model="dslim/bert-base-NER",aggregation_strategy='simple', device_map="auto",dtype=torch.float16)
# example = "My name is Wolfgang and I live in Berlin"
example = "Elon Musk visited Seoul to meet Samsung executives."

for e in ner(example):
    print(e['entity_group'], e['word'], round(e['score'],3))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 504.48it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ORG Elon 0.808
PER Mu 0.504
ORG ##sk 0.685
LOC Seoul 1.0
ORG Samsung 0.999


#### vision-언어모델

In [18]:
vision = pipeline("image-text-to-text", model="HuggingFaceTB/SmolVLM-256M-Instruct", device_map="auto",dtype=torch.float16)
#url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
url = "https://www.biteme.co.kr/blog/wp-content/uploads/2025/01/image-17.jpeg"

messages = [
    {"role": "user", "content": [
        {'type':'image', 'url':url},
        {'type':'text', 'text':'Describe this image in one sentence'},
    ]}
]

print(vision(text=messages, max_new_tokens=40, return_full_text=False)[0]['generated_text'])


Loading weights: 100%|██████████| 471/471 [00:00<00:00, 914.00it/s]


 In this picture we can see a dog wearing a dress and sitting.


In [ ]:
gen = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", device_map="auto",dtype=torch.float16)
messages = [
    {"role": "user", "content": 'HuggingFace를 한 문장으로 설명해 줘'},    
]

print(gen(messages, max_new_tokens=80)[0]['generated_text'][-1]['content'])

Loading weights: 100%|██████████| 338/338 [00:05<00:00, 64.65it/s]
Some parameters are on the meta device because they were offloaded to the disk and cpu.


[{'role': 'user', 'content': 'HuggingFace를 한 문장으로 설명해 줘'}, {'role': 'assistant', 'content': 'Hugging Face는 AI 언어 모델과 라이브러리의 공급 업체로, Transformers와 같은 주요 인공지능 프레임워크의 개발자로서 유명합니다.'}]


In [3]:
from PIL import ImageDraw, Image
import requests


# 웹에서 이미지 다운로드
def download(url):
  return Image.open(requests.get(url,stream=True).raw).convert("RGB")

pipe = pipeline("object-detection", model="facebook/detr-resnet-50")

img = download("https://img.etoday.co.kr/pto_db/2021/07/20210713091336_1643316_1200_918.jpg")
results = pipe(img,threshold=0.8)

draw = ImageDraw.Draw(img)

for r in results:
  b = r['box']
  draw.rectangle([b['xmin'], b['ymin'], b['xmax'], b['ymax']], outline="red", width=3)
  draw.text((b['xmin'],b['ymin'] - 12), f"{r['label']} {r['score']:.2f}", fill="red")

img

ModuleNotFoundError: No module named 'requests'

In [6]:
from transformers import pipeline
import torch
vision = pipeline("image-text-to-text", model="Qwen/Qwen2.5-VL-3B-Instruct", device_map="auto",dtype=torch.float16)

def look(url, question, max_new_tokens=300):
  messages = [
      {"role": "user", "content": [
          {'type':'image', 'url':url},
          {'type':'text', 'text':question},
      ]}
  ]
  out = vision(messages, max_new_tokens=max_new_tokens)
  return out[0]['generated_text'][-1]['content']

Loading weights: 100%|██████████| 824/824 [00:17<00:00, 45.85it/s] 
Some parameters are on the meta device because they were offloaded to the cpu and disk.


ImportError: 
Qwen2VLVideoProcessor requires the Torchvision library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [ ]:
url = 'https://cdn.econoi.com/news/photo/old/news/ND006/1988232117_fRmAurPs_562.jpg'
print(look(url, "공바인다 가격은 얼마야?"))

In [7]:
!pip install -q gTTS

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 2.4.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\soldesk\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [8]:
from gtts import gTTS
from IPython.display import Audio

def speak_gtts(text, path="answer.mp3"):
    gTTS(text, lang="ko").save(path)
    return path

Audio(speak_gtts("안녕하세요. 무엇을 도와드릴까요?"))

ModuleNotFoundError: No module named 'gtts'

In [9]:
!pip install uroman

   ---------------------------------------- 0.0/930.7 kB ? eta -:--:--
   ---------------------------------------- 930.7/930.7 kB 30.7 MB/s  0:00:00



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\soldesk\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip
